# Emulation - Data Space Inversion (DSI)

Everything we have done so far has needed the model. GLM needed it to fill a Jacobian. Monte Carlo needed it once per realisation. iES needed it once per realisation *per iteration*. For the Freyberg model that is fine - it runs in a second. For a real model that takes an hour, it is the whole ballgame.

*Data space inversion* (DSI) takes a different route. Instead of history matching the model, we history match a cheap statistical stand-in - an **emulator** - built from model runs we have already done. The emulator maps the statistical relationships between model outputs: between the outputs we have measurements for, and the outputs we care about forecasting. Conditioning happens in that output ("data") space, not in parameter space.

The consequence is the interesting bit: **DSI needs no new model runs at all.** We already have a prior Monte Carlo ensemble from the iES notebook. That ensemble is all DSI needs.

The trade is just as blunt: we give up parameters. A DSI posterior tells you about forecasts. It cannot tell you what hydraulic conductivity field produced them, because it never had one.

See the original paper [Sun and Durlofsky (2017)](https://doi.org/10.1007/s11004-016-9672-8) and later variants (e.g. [Lima et al 2020](https://doi.org/10.1007/s10596-020-09933-w)). There are also [GMDSI webinars on YouTube](https://www.youtube.com/watch?v=s2g3HaJa1Wk&t=1564s). The [part2 DSI notebook](../part2_10_eva_and_dsi/2_freyberg_ensemble_data_space_inversion.ipynb) goes deeper, on a much higher-dimensional problem.

## How DSI works, briefly

Write $\mathbf{d}$ for the vector of *all* model outputs - the ones we have measurements for and the forecasts, stacked together. A prior Monte Carlo run gives us $N_e$ realisations of it. Centre them and scale by $\sqrt{N_e-1}$:

$$\Delta\mathbf{D} = \frac{1}{\sqrt{N_e-1}}\left[\mathbf{d}_1-\bar{\mathbf{d}},\;\dots,\;\mathbf{d}_{N_e}-\bar{\mathbf{d}}\right]$$

so that $\mathbf{C}_{dd} = \Delta\mathbf{D}\,\Delta\mathbf{D}^T$ is the sample covariance of the outputs - the thing that encodes how every output co-varies with every other, including how the measured quantities co-vary with the forecasts. Take its SVD, $\Delta\mathbf{D} = \mathbf{U}\mathbf{S}\mathbf{V}^T$.

Now the trick. Instead of parameterising the *model*, DSI parameterises the *outputs*:

$$\mathbf{d}(\boldsymbol{\xi}) = \bar{\mathbf{d}} + \mathbf{U}\mathbf{S}\,\boldsymbol{\xi}$$

That is the entire emulator. Feed it a vector $\boldsymbol{\xi}$ of latent coefficients, get back a complete set of model outputs.

Why is that a sensible thing to do? Because if we give $\boldsymbol{\xi}$ a standard normal prior, $\boldsymbol{\xi}\sim N(\mathbf{0},\mathbf{I})$, then

$$\mathrm{Cov}\left[\mathbf{U}\mathbf{S}\boldsymbol{\xi}\right] = \mathbf{U}\mathbf{S}\,\mathbf{I}\,\mathbf{S}^T\mathbf{U}^T = \mathbf{U}\mathbf{S}^2\mathbf{U}^T = \mathbf{C}_{dd}$$

The emulator reproduces the prior covariance of the model outputs *exactly*. That is why the latent parameters get a mean of zero and a standard deviation of one - and it is literally what `prepare_pestpp` writes into `dsi.unc` for us later.

History matching is then: find the $\boldsymbol{\xi}$ values whose measured-quantity entries match our observations. Because $\boldsymbol{\xi}\mapsto\mathbf{d}$ is **linear** and the prior on $\boldsymbol{\xi}$ is **Gaussian**, this is the linear-Gaussian Bayesian problem - the same one FOSM solves, but posed in output space instead of parameter space. `pestpp-ies` solves it iteratively, and because it is iterative it copes with the mild nonlinearity that transformations introduce (more on those later).

Two things to notice, because both come back to bite us:

1. The model appears nowhere in that equation. Once $\mathbf{U}$ and $\mathbf{S}$ are computed, MODFLOW is done.
2. Everything rests on $\mathbf{C}_{dd}$ - a mean and a covariance. Those two moments pin down a distribution **only if that distribution is Gaussian**. Hold that thought.

## The Current Tutorial

In this notebook we will:
1. Reuse the prior Monte Carlo ensemble from the iES notebook as training data - and screen it, because it needs it
2. Build a DSI emulator with `pyemu.emulators.DSI`
3. Condition it with `pestpp-ies`, using the same observations, weights and phi as every other part1 notebook
4. Compare the DSI forecast posterior against the iES forecast posterior
5. Be honest about what DSI cannot do

### Admin

> **This notebook needs the iES notebook to have been run first.** We use its prior observation ensemble as our training data, so run [part1_13](../part1_13_basic_ies/freyberg_ies.ipynb) before this one. We do *not* build or run the model here - that is rather the point.

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)

import time
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt;

import pyemu
import flopy
assert "dependencies" in flopy.__file__
assert "dependencies" in pyemu.__file__
sys.path.insert(0,"..")
import herebedragons as hbd

plt.rcParams['font.size'] = 10
pyemu.plot_utils.font = 10

## The training data

DSI needs an ensemble of model outputs that spans **both** the quantities we have measurements for **and** the forecasts we care about. That is exactly what a prior Monte Carlo run produces, and it is the only time the model has to run.

We already paid for one in the iES notebook: the prior ensemble, iteration 0 of the first `pestpp-ies` run. Let's go and get it.

In [ ]:
# the iES notebook's master directory - we are borrowing its results, not re-running anything
ies_d = os.path.join('..','part1_13_basic_ies','master_ies')
assert os.path.exists(ies_d), "run the part1_13 iES notebook first!"

pst = pyemu.Pst(os.path.join(ies_d,'freyberg_pp.pst'))

# iteration 0 is the prior - the model outputs before any data was assimilated
oe = pd.read_csv(os.path.join(ies_d,'freyberg_pp.0.obs.csv'),index_col=0)
oe.columns = [c.lower() for c in oe.columns]
oe = oe.astype(float)
oe.shape

So we have an ensemble of model outputs: one row per realisation, one column per observation in the control file. Note what is in there and what it cost us:

In [ ]:
print(f'realisations in the training data : {oe.shape[0]}')
print(f'model outputs per realisation     : {oe.shape[1]}')
print(f'non-zero weighted observations    : {pst.nnz_obs}')
print(f'observation groups being fitted    : {sorted(pst.nnz_obs_groups)}')
print(f'forecasts                         : {pst.forecast_names}')
print()
print('new model runs needed for DSI      : 0')

## First, screen the training data

It is tempting to go straight to the emulator. Don't.

DSI works by taking the principal components of the training data. Principal components are driven by **variance**, and variance is not robust: one output with a silly value in one realisation can swamp everything else. And a prior Monte Carlo run on a groundwater model *will* contain silly values - realisations where a cell went dry, or where the solver did not really converge.

Heads in this model are a few tens of metres. Let's see if that is what we actually have.

In [ ]:
head_obs = [o for o in oe.columns if o.startswith('trgw')]

print('range of simulated heads across the whole prior ensemble:')
print(f'  min: {oe[head_obs].values.min():.3g}')
print(f'  max: {oe[head_obs].values.max():.3g}')

Ouch. A head of -1e15 m is not a water level, it is a model that fell over. Let's find out how widespread it is, and which outputs are affected.

In [ ]:
# a negative head is unambiguous garbage - there is no reading of this model in
# which a water level is -1e15 m
negative = (oe[head_obs] < 0).any(axis=1)
print(f'realisations with a negative head: {negative.sum()}')

# the high end is less clear cut. Heads at the observation wells sit around
# 30-40 m, but the ensemble tails off smoothly - there is no obvious break
print('\nper-realisation maximum head, quantiles:')
print(oe[head_obs].max(axis=1).quantile([0.5,0.9,0.95,0.99,1.0]).round(1).to_string())

# which sites go bad, and how badly
worst = oe[head_obs].min().sort_values()
print('\nworst five outputs (minimum value over the ensemble):')
print(worst.head(5).to_string())

Now the part that matters. Is any of this contaminating something we care about?

In [ ]:
# are the observations we are fitting affected? and the forecasts?
affected = worst[worst < 0].index.tolist()
print('contaminated outputs that are weighted observations:')
print(' ', [o for o in affected if o in pst.nnz_obs_names] or 'none')
print('contaminated outputs that are forecasts:')
print(' ', [o for o in affected if o in pst.forecast_names] or 'none')

It is in both. Twelve of the observations we are *fitting* are contaminated - all at the `trgw-0-26-6` site - and so is `trgw-0-9-1:4383.5`, the groundwater level forecast we have been tracking since the trial-and-error notebook. These are not realisations that are slightly off; they are numerical garbage.

Worth pausing on, because none of it is a DSI problem. It was sitting in the prior ensemble the whole time, quietly skewing the prior histograms back in the Monte Carlo and iES notebooks. DSI just makes it impossible to ignore, because of what variance does to principal components. Here is the damage, measured as the share of total ensemble variance sitting in a single output:

In [ ]:
def cumulative_energy(data):
    """the share of ensemble variance captured by each successive principal
    component - the same calculation DSI uses internally when deciding how
    many components to keep"""
    X = data.astype(float)
    z = (X - X.mean()) / np.sqrt(X.shape[0] - 1)
    s = np.linalg.svd(z.values, full_matrices=False)[1]
    return np.cumsum(s**2) / np.sum(s**2)


def n_components_for(ce, threshold):
    """how many components are needed to reach `threshold` of the variance"""
    return int(np.argmax(ce >= threshold) + 1)


ce_raw = cumulative_energy(oe)
print(f'biggest single output as a share of total variance: '
      f'{100*oe.var().max()/oe.var().sum():.1f}%')
print()
print('components needed, using the raw ensemble:')
for t in [0.9, 0.99, 0.999]:
    print(f'  {t*100:5.1f}% of variance -> {n_components_for(ce_raw, t)} component(s)')

One component holds 99.9% of the "variance". That is not a low-dimensional problem, it is a broken one - the PCA is describing a single blown-up number and nothing else. An emulator fitted to this would be worthless.

So we screen. We drop the offending realisations and keep the rest.

In [ ]:
# drop the unambiguous failures, plus realisations whose heads climb implausibly
# high. The 100 m cut is OUR CHOICE - there is no break in the data to guide us,
# so this is a judgement about what is physically plausible. Change it and see
MAX_PLAUSIBLE_HEAD = 100.0
blown_up = negative | (oe[head_obs] > MAX_PLAUSIBLE_HEAD).any(axis=1)

data = oe.loc[~blown_up, :]
print(f'training data: {data.shape[0]} realisations (dropped {blown_up.sum()})')
print(f'head range now: {data[head_obs].values.min():.2f} to {data[head_obs].values.max():.2f} m')

ce = cumulative_energy(data)
print('\ncomponents needed, after screening:')
for t in [0.9, 0.99, 0.999]:
    print(f'  {t*100:5.1f}% of variance -> {n_components_for(ce, t)} component(s)')

Much healthier. The variance is now spread over many components, which is what we would expect from an ensemble that genuinely varies in a lot of directions.

> **Lesson, and it is the most practical one in this notebook:** an emulator is only ever as good as the ensemble it was trained on. Screening the training data is not optional housekeeping, it is part of the method.

Notice how little of that screening was objective. The negative heads decided themselves. The 100 m cut did not - the ensemble tails off smoothly, so we drew a line based on what we think a water level in this aquifer can plausibly be. Draw it at 60 m and you drop 44 realisations instead of 20; draw it at 200 m and you keep realisations that are probably junk. Like the streamflow weight back in part1_01, it is a subjective call that changes the answer, so make it deliberately and write it down.

We also threw away whole realisations rather than individual outputs. That is another choice - we could have dropped the `trgw-0-24-4` site entirely and kept its realisations. Which is better depends on whether you need that site.

There is a second scale problem hiding underneath this one, though, and screening does not touch it. Look at where the variance actually lives now:

In [ ]:
# how much of the ensemble variance does each observation group carry?
groups = pst.observation_data.groupby('obgnme').obsnme.apply(list)
rows = []
for grp, names in groups.items():
    names = [n for n in names if n in data.columns]
    if len(names) == 0:
        continue
    rows.append((grp, len(names), data[names].var().sum()))
vardf = pd.DataFrame(rows, columns=['group','nobs','total_variance'])
vardf['share_%'] = 100 * vardf.total_variance / vardf.total_variance.sum()
vardf.sort_values('share_%', ascending=False).round(2)

Streamflow and particle travel time carry essentially all of it. Every head site rounds to **0.0%** - and heads are half of the observations we are actually fitting.

This is not because heads are uninformative. It is because an SVD ranks directions by *absolute* variance and has no idea that one column is in metres and another is in cubic metres per day. Head variance across this ensemble is around $10^3$; streamflow variance is around $10^8$. The components will describe streamflow and ignore heads, whatever we do with weights later.

So the emulator we are about to build is, in effect, a model of the flows. We will come back and fix this once we have seen what else needs fixing.

## Building the emulator

`pyemu.emulators` is the entry point for all things emulation. To build a `DSI` object we need the training data. Optionally we pass a `Pst`, which `DSI` uses later to help build a PEST interface for the emulator, plus transformations and an SVD truncation level.

We will keep `energy_threshold=1.0`, which means no truncation - keep every component. For a problem this size that costs nothing.

In [ ]:
from pyemu.emulators import DSI

dsi = DSI(pst=pst,               # optional; used to build the PEST interface later
          data=data,             # the training data - this is the required bit
          transforms=None,       # optional; see the limitations discussion below
          energy_threshold=1.0)  # 1.0 = keep every component

dsi.fit();

`fit()` did the work: centre the training data, take its SVD, and store the projection that turns a vector of latent coefficients back into a full set of model outputs.

The singular values tell us how much each component carries:

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(9,3.5))
axes[0].plot(dsi.s,'b-')
axes[0].set_yscale('log')
axes[0].set_xlabel('component')
axes[0].set_ylabel('singular value')
axes[0].set_title('singular value spectrum')

axes[1].plot(ce,'b-')
axes[1].axhline(0.99,color='r',ls='--',lw=1.0,label='99% of variance')
axes[1].set_xlabel('number of components')
axes[1].set_ylabel('cumulative share of variance')
axes[1].set_title('cumulative energy')
axes[1].legend()
plt.tight_layout();

## What are the parameters now?

This is the conceptual jump. Our model had 68 adjustable parameters: pilot point hydraulic conductivities and recharge. The emulator has none of them. Its "parameters" are the **latent coefficients** - the weights on each principal component of the output ensemble.

To run the emulator, hand it a vector of those coefficients. That is the entire forward run:

In [ ]:
print(f'the emulator takes a vector of {dsi.s.shape[0]} latent coefficients')

pvals = np.random.normal(0, 1, dsi.s.shape)
sim = dsi.predict(pvals)

print(f'and returns {sim.shape[0]} simulated model outputs')
sim.head()

No MODFLOW. No files. A matrix multiply.

Notice what this buys and what it costs. Any vector of coefficients gives you a complete, self-consistent set of model outputs - heads, flows, travel time, historic and forecast - in microseconds. But there is no hydraulic conductivity field anywhere in that answer, and no way to get one back out.

## A PEST interface for the emulator

We now want `pestpp-ies` to adjust those latent coefficients until the emulated outputs match our measurements. That needs a control file, template files and instruction files - and `prepare_pestpp()` builds all of it.

We pass `use_runstor=True`, which sets the emulator up to be driven through PEST++'s **external run manager**. More on why in a moment.

In [ ]:
t_d = 'dsi_template'
pst_dsi = dsi.prepare_pestpp(t_d=t_d, use_runstor=True)

# prepare_pestpp builds a fresh control file, so the pestpp options from the model
# don't come across. We very much want our forecasts to travel with us
pst_dsi.pestpp_options['forecasts'] = pst.pestpp_options['forecasts']
assert pst_dsi.forecast_names == pst.forecast_names

# and it doesn't copy binaries over either - note we need pestpp-ies, but no MODFLOW
hbd.prep_bins(t_d)

sorted(os.listdir(t_d))

`dsi.pickle` is the fitted emulator. `dsi_pars.csv` holds the latent coefficients, `dsi_sim_vals.csv` the emulated outputs, and `forward_run.py` is the "model" - it loads the emulator and calls `predict()`.

Let's see how the interface changed:

In [ ]:
print(f'{"":22s}{"model":>10s}{"emulator":>10s}')
print(f'{"adjustable parameters":22s}{pst.npar_adj:>10d}{pst_dsi.npar_adj:>10d}')
print(f'{"observations":22s}{pst.nobs:>10d}{pst_dsi.nobs:>10d}')
print(f'{"weighted observations":22s}{pst.nnz_obs:>10d}{pst_dsi.nnz_obs:>10d}')

The observations are identical - same 36 weighted observations, same groups, same weights. That is deliberate, and it is what makes this notebook comparable to the ones before it: **phi means the same thing here as it did in the GLM and iES notebooks.** Whatever number we end up with can be read against those.

The parameters are a different animal entirely: latent coefficients instead of pilot points.

### Guard against extrapolation

An emulator interpolates between the runs it was trained on. Ask it to extrapolate and it will answer confidently and wrongly.

The specific way this bites in DSI is *prior data conflict*: an observation whose measured value lies outside the range the training ensemble ever produced. There is no combination of latent coefficients that will reach it, so `pestpp-ies` will contort the whole ensemble trying. Better to detect those observations and drop them, which is what `ies_drop_conflicts` does. Turn it on - for DSI it should be the default.

In [ ]:
pst_dsi.pestpp_options["ies_drop_conflicts"] = True

## Conditioning the emulator

Now we run `pestpp-ies` on the emulator instead of on the model.

One new thing here. Normally we deploy a swarm of PANTHER workers, each running the model in its own directory. That is the right design when the model takes minutes, but it is entirely wrong for DSI: the emulator run takes microseconds, so all the time goes into starting python and reading files. Deploying workers would make DSI a hundred times slower than it needs to be.

Instead we use PEST++'s **external run manager**, with the `/e` switch. It works batch-wise: PEST++ writes every parameter set it wants evaluated into a run storage file (`dsi.rns`), calls the forward run **once**, and reads all the results back. So `dsi.predict()` gets called a single time per iteration, on the whole ensemble at once, in one process. No workers, no ports, no worker directories.

In [ ]:
# as always, as many realisations as you can afford. Here that is all of them
pst_dsi.pestpp_options["ies_num_reals"] = data.shape[0]
pst_dsi.control_data.noptmax = 3
pst_dsi.write(os.path.join(t_d,"dsi.pst"),version=2)

# run in a copy, so the template folder stays clean
m_d = 'master_dsi'
if os.path.exists(m_d):
    shutil.rmtree(m_d)
shutil.copytree(t_d, m_d)

start = time.time()
pyemu.os_utils.run("pestpp-ies dsi.pst /e", cwd=m_d)
print(f'\nconditioning took {time.time()-start:.1f} seconds')

Seconds. For comparison, the iES run in the previous notebook needed 50 model runs per iteration, and the prior Monte Carlo before it needed 250.

Let's look at the objective function. Remember this is the *same* phi - same observations, same weights - that we have been minimising since the trial-and-error notebook.

In [ ]:
pst_dsi = pyemu.Pst(os.path.join(m_d,"dsi.pst"))
phidf = pd.read_csv(os.path.join(m_d,"dsi.phi.actual.csv"))

fig,ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(phidf.iteration, phidf['mean'], "bo-", label='ensemble mean phi')
ax.fill_between(phidf.iteration, phidf['min'], phidf['max'],
                color='b', alpha=0.15, label='ensemble min/max')
ax.axhline(pst_dsi.nnz_obs, color='r', ls='--', lw=1.5,
           label=f'nnz_obs = {pst_dsi.nnz_obs}')
ax.set_yscale('log')
ax.set_xlabel('iteration')
ax.set_ylabel('$\\Phi$')
ax.legend()
plt.tight_layout();

Phi drops hard, and then keeps going - straight past the number of non-zero weighted observations (the red line).

That red line is a rough overfitting alarm. If weights were set as the inverse of the standard deviation of measurement noise, then a model fitting the data *as well as the noise allows* should land at a phi of roughly `nnz_obs`. Going well below it means we are fitting noise.

But look closely at what our weights actually are. Throughout part1 we have used `HEAD_WEIGHT = 1.0` and `SFR_WEIGHT = 0.003` - numbers we chose for balance and visibility back in the trial-and-error notebook, not measurements of anything. They are not inverse noise standard deviations. `pestpp-ies` even told us as much in the record file: with no noise information available it fell back to `ies_no_noise`, so there is no noise floor for phi to respect.

So we cannot actually tell, from this run, whether we are overfitting. That is not a DSI flaw - it is the bill arriving for a subjective weighting choice made twelve notebooks ago. Part2 does this properly, assigning weights as `1/standard_deviation` so that the `nnz_obs` line means something.

## Did it fit?

Same stochastic 1-to-1 plot we have used since the iES notebook, so you can flip between the two. Prior in grey, posterior in blue.

In [ ]:
oe_pr_dsi, oe_pt_dsi = hbd.load_ies_obs_ensembles(m_d, case='dsi')
hbd.plot_1to1_ensemble(pst_dsi, prior=oe_pr_dsi, posterior=oe_pt_dsi,
                       title='DSI emulator');

The emulator fits the measured data well - unsurprising, given the phi history. The question that actually matters is whether it gets the *forecasts* right.

## Predictive uncertainty analysis

This is what we came for. Prior in grey, posterior in blue, and the truth as a dashed black line. Remember we only know the truth because this is a synthetic problem - and remember that a forecast is a success only if the posterior *brackets* the truth, not if it hits it exactly.

In [ ]:
for forecast in pst_dsi.forecast_names:
    fig,ax = plt.subplots(1,1,figsize=(5,2.8))
    oe_pr_dsi.loc[:,forecast].hist(bins=20,alpha=0.5,color="0.5",ax=ax,label='prior')
    oe_pt_dsi.loc[:,forecast].hist(bins=20,alpha=0.5,color="b",ax=ax,label='posterior')
    v = pst_dsi.observation_data.loc[forecast,"obsval"]
    ax.axvline(v,color='k',ls='--',lw=2.0,label='truth')
    ax.set_ylabel('count')
    ax.set_title(forecast)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## Hold on - are these answers even possible?

Before we celebrate, let's ask something the fit statistics cannot tell us: are the numbers the emulator produced physically possible at all?

Streamflow in a river cannot be negative. Neither can a particle travel time. Let's count.

In [ ]:
def check_feasible(oe_post, label):
    """count values the emulator produced that cannot physically happen"""
    gage = [n for n in groups['gage-1'] if n in oe_post.columns]
    neg = (oe_post[gage] < 0)
    ptime = oe_post['part_time']
    print(f'[{label}]')
    print(f'  negative streamflow  : {int(neg.values.sum())} values, in '
          f'{int(neg.any(axis=1).sum())} of {oe_post.shape[0]} realisations')
    print(f'  lowest streamflow    : {oe_post[gage].values.min():,.1f} m3/d')
    print(f'  negative travel times: {int((ptime<0).sum())}, '
          f'lowest {ptime.min():,.1f} days')


check_feasible(oe_pt_dsi, 'untransformed')

That is not a rounding problem. A large fraction of our posterior realisations contain rivers flowing backwards at thousands of cubic metres a day, and some contain particles that arrive before they set off.

Every one of those realisations went into the forecast histograms above.

## Why this happens: the normality assumption

Go back to the emulator equation:

$$\mathbf{d}(\boldsymbol{\xi}) = \bar{\mathbf{d}} + \mathbf{U}\mathbf{S}\,\boldsymbol{\xi}, \qquad \boldsymbol{\xi}\sim N(\mathbf{0},\mathbf{I})$$

We showed that this reproduces $\mathbf{C}_{dd}$ exactly. What we glossed over is that a mean and a covariance **do not describe a distribution** - unless that distribution happens to be Gaussian. The multivariate normal is the one family that is completely determined by its first two moments. For anything else, matching the mean and covariance matches two summary statistics and throws the rest away.

So what does DSI actually generate? A linear combination of Gaussian variables is Gaussian. Which means $\mathbf{d}(\boldsymbol{\xi})$ is **multivariate normal, by construction** - regardless of what our prior ensemble looked like. If the outputs really were Gaussian, that is exact and free. If they were not, DSI has quietly replaced their distribution with the Gaussian that shares its mean and covariance.

That substitution has a specific and damaging consequence: **a Gaussian has infinite support in every direction.** It puts probability mass everywhere from $-\infty$ to $+\infty$. Nothing in the algebra knows that streamflow stops at zero, or that travel time does. So it strays past the bounds, and the negative flows above are exactly that happening.

Are our outputs Gaussian? Not remotely - and we can measure it. Skewness is zero for any symmetric distribution:

In [ ]:
from scipy.stats import skew

print('how skewed are the training-data marginals? (0 = symmetric)')
for grp in ['gage-1','particle','headwater','trgw-0-3-8']:
    names = [n for n in groups[grp] if n in data.columns]
    sk = np.abs(skew(data[names].values, axis=0))
    print(f'  {grp:12s} mean |skew| = {sk.mean():5.2f}   worst = {sk.max():5.2f}')

Strongly skewed, all of them - which is entirely normal for groundwater model outputs. Fluxes and travel times are bounded below and have long right tails; that is what they *are*.

## Transformations: getting closer to normal

We cannot make the outputs Gaussian. But we can history match a *transformed* version of them that is much closer to Gaussian, and then map back. If $g$ is monotonic and invertible, doing DSI on $g(\mathbf{d})$ and reporting $g^{-1}$ of the result is a legitimate change of variables - the SVD, the standard-normal prior and the linear-Gaussian update all happen in transformed space, where the assumption is much less of a lie.

Two transforms do most of the work, and they solve two different problems.

**`log10` - for strictly positive, right-skewed quantities.** The log of a lognormal variable is *exactly* normal, and plenty of hydrologic quantities are roughly lognormal. It also fixes feasibility for free, and structurally rather than by luck: the inverse transform is $10^x$, which is positive for every real $x$. A log-transformed output *cannot* come back negative, no matter what the latent coefficients do.

**`normal_score` - for anything at all.** Rank every value of an output within its own ensemble, then map that rank to the matching quantile of a standard normal. By construction the transformed marginal is *exactly* $N(0,1)$ - whatever shape it started with: skewed, bounded, bimodal, doesn't matter. The inverse maps back through the empirical distribution, so returned values stay inside (roughly) the range the training ensemble spanned. Feasibility again comes for free.

It also fixes the variance-scale problem we found earlier. Once every output has been mapped to $N(0,1)$, they all have unit variance, so no group can dominate the SVD by virtue of its units. Heads get a say again.

### But know what you are transforming

This is where it is easy to do real damage. "Log the fluxes" is wrong advice here, and the model will not warn you:

In [ ]:
for grp in ['gage-1','headwater','tailwater','particle']:
    names = [n for n in groups[grp] if n in data.columns]
    v = data[names].values
    print(f'{grp:11s} min = {v.min():12,.1f}   values <= 0: {int((v<=0).sum()):5d} of {v.size}')

- `part_time` is strictly positive. `log10` is a good fit.
- `gage-1` is non-negative but hits **exactly zero** - the river dries up in some realisations. $\log_{10}(0)$ is undefined, so a plain log breaks. You would need an offset, or a different transform.
- `headwater` and `tailwater` are **negative most of the time**, and rightly so: they are groundwater/surface-water *exchange* fluxes, where the sign tells you which way the water goes. Log-transforming them is not a numerical inconvenience, it is a category error.

So: `log10` on the travel time, `normal_score` on everything else. The transforms are applied in the order listed.

In [ ]:
transforms = [
    # strictly positive and skewed - and log10 guarantees it comes back positive
    {'type':'log10', 'columns':['part_time']},
    # everything else, whatever its shape or sign. Also puts every output on the
    # same footing, so heads are no longer swamped by flows in the SVD
    {'type':'normal_score'},
]

Now rebuild and re-condition. These are the same steps we just worked through, wrapped in a function so we can run them again.

In [ ]:
def build_and_condition(transforms, tag, noptmax=3):
    """fit a DSI emulator, build a PEST interface for it, and condition it -
    exactly the steps we did by hand above"""
    dsi = DSI(pst=pst, data=data, transforms=transforms, energy_threshold=1.0)
    dsi.fit()

    t_d = f'template_{tag}'
    p = dsi.prepare_pestpp(t_d=t_d, use_runstor=True)
    p.pestpp_options['forecasts'] = pst.pestpp_options['forecasts']
    hbd.prep_bins(t_d)
    p.pestpp_options['ies_drop_conflicts'] = True
    p.pestpp_options['ies_num_reals'] = data.shape[0]
    p.control_data.noptmax = noptmax
    p.write(os.path.join(t_d,'dsi.pst'), version=2)

    m_d = f'master_dsi_{tag}'
    if os.path.exists(m_d):
        shutil.rmtree(m_d)
    shutil.copytree(t_d, m_d)
    pyemu.os_utils.run("pestpp-ies dsi.pst /e", cwd=m_d)
    return dsi, m_d


dsi_tx, m_d_tx = build_and_condition(transforms, 'tx')
pst_tx = pyemu.Pst(os.path.join(m_d_tx,'dsi.pst'))
oe_pr_tx, oe_pt_tx = hbd.load_ies_obs_ensembles(m_d_tx, case='dsi')

And the question that started this section:

In [ ]:
check_feasible(oe_pt_dsi, 'untransformed')
print()
check_feasible(oe_pt_tx, 'transformed')

Gone. Not reduced - gone, and gone by construction rather than by good fortune. The lowest streamflow the transformed emulator can produce is the lowest one in the training ensemble, because that is what the inverse normal-score transform maps back to.

Let's also check we did not wreck the fit to get here:

In [ ]:
for tag, m_d in [('untransformed','master_dsi'), ('transformed', m_d_tx)]:
    phi = pd.read_csv(os.path.join(m_d,'dsi.phi.actual.csv'))
    print(f'{tag:14s} phi: {phi["mean"].iloc[0]:8.1f} -> {phi["mean"].iloc[-1]:6.2f}')

Both fit the data comfortably. The difference is that one of them produced answers that could actually happen.

Here are the forecasts from the transformed emulator - the ones we should have been looking at all along:

In [ ]:
for forecast in pst_tx.forecast_names:
    fig,ax = plt.subplots(1,1,figsize=(5,2.8))
    oe_pr_tx.loc[:,forecast].hist(bins=20,alpha=0.5,color="0.5",ax=ax,label='prior')
    oe_pt_tx.loc[:,forecast].hist(bins=20,alpha=0.5,color="b",ax=ax,label='posterior')
    v = pst_tx.observation_data.loc[forecast,"obsval"]
    ax.axvline(v,color='k',ls='--',lw=2.0,label='truth')
    ax.set_ylabel('count')
    ax.set_title(f'{forecast}  (transformed)')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

> **One honest caveat.** A normal-score transform makes each output's *marginal* distribution exactly normal. It does **not** make the *joint* distribution multivariate normal - that would need the dependence between outputs to be linear too, and it generally is not. So transforms get us much closer to the assumption DSI needs, and they buy us physical feasibility outright, but they do not make the assumption true. Nonlinear relationships between observations and forecasts still get smeared.

### How does that compare to iES?

DSI and iES are answering the same question from the same prior ensemble. iES did it by adjusting model parameters and re-running the model. DSI did it by adjusting latent coefficients and never touching the model. If they agree, that is a strong argument for using the cheap one.

Let's put them side by side. The iES posterior is the last iteration of the second run in the previous notebook.

In [ ]:
# the iES posterior from the previous notebook
ies_d1 = os.path.join('..','part1_13_basic_ies','master_ies1')
oe_pt_ies = pd.read_csv(os.path.join(ies_d1,'freyberg_pp.5.obs.csv'),index_col=0)
oe_pt_ies.columns = [c.lower() for c in oe_pt_ies.columns]

fig,axes = plt.subplots(1,len(pst_tx.forecast_names),
                        figsize=(4*len(pst_tx.forecast_names),3.2))
for ax,forecast in zip(axes,pst_tx.forecast_names):
    ax.hist(oe_pt_ies.loc[:,forecast].values,bins=15,alpha=0.5,
            color='g',density=True,label='iES posterior')
    ax.hist(oe_pt_tx.loc[:,forecast].values,bins=15,alpha=0.5,
            color='b',density=True,label='DSI posterior')
    v = pst_dsi.observation_data.loc[forecast,"obsval"]
    ax.axvline(v,color='k',ls='--',lw=2.0,label='truth')
    ax.set_title(forecast)
    ax.set_yticks([])
    ax.legend(fontsize=8)
plt.tight_layout();

Have a good look at these. Where the two agree, DSI has given us the same answer for a tiny fraction of the compute. Where they disagree, the interesting question is *which one to believe* - and note that neither is automatically right. The two runs also used different ensemble sizes, so some of the difference is just sampling.

## Limitations

DSI is fast, it is easy to run, and it is genuinely useful. It is not magic, and the ways it fails are not always loud. In rough order of how often they bite:

**1. It cannot go outside its training data.** The emulator is a weighted recombination of the ensemble it was fitted to. If no realisation ever produced a low enough streamflow, no combination of latent coefficients will produce one either. When the measured data sits outside that envelope you get prior data conflict - which is why `ies_drop_conflicts` should always be on, and why a conflict is a message about your prior, not a nuisance to be silenced. If you need to go somewhere the ensemble never went, the fix is more model runs, not a cleverer emulator.

**2. Garbage in, garbage out - and it is worse than usual.** We spent the first third of this notebook on this for good reason. Because DSI is built on variance, one blown-up realisation does not just add noise, it can consume the entire latent space. Always look at your training data before you fit.

**3. Untransformed, the whole thing rests on a normality assumption it does not meet.** This is the one we spent most of the notebook on. The reparameterisation reproduces a mean and a covariance, which describes a distribution only if it is Gaussian - and a Gaussian has infinite support, so the emulator cheerfully produced negative streamflow in 42% of realisations. On top of that, an SVD ranks directions by absolute variance and does not know metres from m³/d, so head observations contributed 0.0% of the components before we transformed. Transformations address both, and `normal_score` addresses them at the same time. They are not an optional refinement - treat an untransformed DSI run as a first sketch, not an answer. But pick them for what the quantity physically is: `log10` on a signed exchange flux is a category error, not a tuning choice. `rowwise_groups` is the other lever, scaling each observation group before the SVD.

**4. No parameters means no parameter insight.** You cannot ask a DSI posterior which part of the aquifer is transmissive, look at a posterior hydraulic conductivity field, or check whether a realisation is physically sensible. If your question is "what is the forecast and how uncertain is it", that is fine. If your question is "why", DSI cannot answer it.

**5. Conditioning is linear-Gaussian in the latent space, and transforms only get you closer.** DSI assumes the relationship between observations and forecasts is captured by the covariance of the training ensemble. A normal-score transform makes each output's marginal exactly normal, but it does not make the joint distribution multivariate normal, so nonlinear dependence between observations and forecasts still gets smeared. One visible symptom is emulated time series that go saw-toothed in the forecast period - smooth where there is data to pin them down, jumping around where there is not. Part2 shows this clearly.

**6. It inherits every assumption in your prior.** DSI conditions the prior ensemble; it does not question it. If the prior parameterisation is too coarse - as we found out the hard way in the Monte Carlo and regularisation notebooks - DSI will hand you a posterior that is too narrow, with exactly the same confidence as a good one. It is not a way to escape a badly posed problem.

**7. There is no free lunch on the model runs.** DSI moves the cost, it does not remove it. You still need a prior Monte Carlo ensemble big enough to span the behaviour you care about, and that ensemble is the expensive part. What DSI buys you is the ability to condition it - and re-condition it, with different data, different weights, different scenarios - for almost nothing.

## So where does that leave us?

DSI is the first method in these tutorials that decouples the cost of history matching from the cost of running the model. For the Freyberg model that is a curiosity. For a model that takes an hour a run, it is the difference between an uncertainty analysis and no uncertainty analysis.

The catch is that it buys speed with assumptions, and it is quiet about them. Everything in the limitations list above will hold whether or not you check for it.

Next steps:
- [part2_10](../part2_10_eva_and_dsi/2_freyberg_ensemble_data_space_inversion.ipynb) does DSI on the high-dimensional part2 problem, with transformations, noise-based weights and time-series diagnostics
- the same folder has notebooks on ensemble data worth and on PLS emulation, for comparison